# A Neural Probabilistic Language Model Implementation
(2003 Bengio et Al)

This notebook will attempt to teach you about Word Embeddings and the problems they solve

## What are Word Embeddings & Why are they useful

Word Embeddings are more straightforward than you think. They are simply a mathematical way to represent words, usually in the form of a N-dimesional vector or tensor\
For example, we could represent the word dog as a 3D vector [1, 2, 3] or notebook as [4, 5, 6]\
Each dimension in the vector can represent certain aspects and ways to describe the word.\
In the case of the word dog, the 1st dimension could represnt it's a living organism, the 2nd dimension could represent it's an animal, the 3rd that it's a pet\
Note that in reality we won't be able to point to a dimesnion and say "It represents this!", it is just an example

One of the goals of our model is to group together words that have similar semantic and grammatical meanings.\
For example the words dog and cat should have similar vectors (known as feature vectors) because they are used in similar contexts and scenarios.\
"I have a pet dog" or "I have a pet cat"\
Dog = [1, 2, 3]\
Cat = [1.2, 2.3, 2.9]

### Why are the useful

Word Embeddings help with a few things.\
One important thing they help with is that all neural networks need numerical inputs, you cannot forward pass a word into a neural net, so word embeddings help solve this problem. One thing to note is that this problem has already been solved using another technique I will get into right now.\
\
Previously, to represent words we would need to one hot encode each word, that is each word would be assigned an N dimesion vector (N = entire vocab) where a certain dimesion is set to 1 and the rest of the dimesions are set to 0. For this we would have to create a NxN identity matrix. N can get very large and the input layer (depending on the vocab and context length) can exceed 100000+ neurons. This is very computationally and memory expensive and doesn't allow for semantically similar words to be grouped together.

But the biggest setback with this approach is that we can't group together similar words. The words 'dog' and 'cat' are equally the same as the words 'cat' and 'calculator' because each word is it's own dimension.

Word embeddings help solve this problem by densely packing each word in to a lower dimesional space (known as the embedding space). This space can vary in size depending on the vocab size but it usually spans anywhere from 30 - 1000 dimesnional space which is far less than the N > 10000 dimesional space. Here the model can learn and adjust these embeddings to group together similar words.

### What are we trying to actually do

This model's main goal is to predict the next word given the previous n words (we will use the previous 3 words for now as the paper uses 3 words)./
But what about the embeddings?\
The model will also learn good ways to represent semantically similar words almost as a byproduct of trying to correctly predict the next word.

How is this possible?
The model will see patterns the text like "I have a pet __", "Pet ___ are very common" and it will eventually learn and be rewarded to group together semantically similar words

In [1]:
#Just some important imports
import torch
import torch.nn as nn
import math
import numpy as np

In [2]:
#Some constants I might use
HIDDEN_LAYER = 700

In [3]:
#Get the dataset/massive hunk of text
#The text is already clean and is the Penn TreeBank (PTB) dataset, very famous dataset for NLP tasks
raw_text = open('../data/text_chunk.txt', 'r').read()
words = raw_text.split()

In [4]:
#Now from words we need to identify the unique words/vocabulary within the text
#This will be our vocabulary size
unique_words = sorted(list(set(words)))
vocab_size = len(unique_words)

In [5]:
#We want to create mapping of each value in unique words to an index/value both ways
#This will be useful later when we want to look up the words embedding vector
ix_to_words = {i:word for i, word in enumerate(unique_words)}
words_to_ix = {word:i for i, word in ix_to_words.items()}

In [6]:
#Here we will intialize the embeddings matrix we will do 128D embeddings
embedding_space = 128
context_length = 3                            # This is how many previous words we will use to predict the next word

#We will use uniform instead of normal distrubtion but with range from [-0.1, 0.1], both normal and uniform work as long as embedding values are not
#too large which can cause signal blow up
C = torch.rand((vocab_size, embedding_space)) * 0.2 - 0.1
print(C.shape)

torch.Size([9999, 128])


In [7]:
#Now we need to actually create our train and test set
#What this looks like is kind of a sliding window where we take the first three preceding words and put it into the training set
#     The current word (word that comes after the 3 previous word) will go into the target set
#     We will not directly embed the words, rather we will insert the index correlated with the word into the training & target sets

print(f'Text Length: {len(words)}')

X = torch.zeros((len(words) - 3, 3))
Y = torch.zeros(len(words) - 3)

for i in range(3, len(words)):
    training_instance = torch.tensor([words_to_ix[words[i-3]], words_to_ix[words[i-2]], words_to_ix[words[i-1]]])
    X[i-3] = training_instance
    Y[i-3] = words_to_ix[words[i]]
    
print('Created Training & Label Sets \n')

Text Length: 887521
Created Training & Label Sets 



In [8]:
#We will create train, validation, and test sets now, they will be 80%, 10%, 10% of all present data respectively
train_len = len(X) // 10 * 8
val_len = len(X) // 10

X_train = X[:train_len]
Y_train = Y[:train_len]

X_val = X[train_len:train_len + val_len]
Y_val = Y[train_len:train_len + val_len]

X_test = X[train_len + val_len:]
Y_test = Y[train_len + val_len:]

Layers and Neurons:\
     - First/Input Layer: 384 Neurons, naturally comes from embedding_space = 128 and using context length = 3\
     - Hidden Layer: 700 Neurons, I feel like good size because not too big which will cause overfitting, I should expirement with this more though\
     - Output Layer: 9999 Neurons, this matches the number of vocab_words which is what we are trying to predict, the next word\

In [9]:
#Just doing some shape checking, VERY IMPORTANT
#len(words) x 3
print(X.shape)
#len(words)
print(Y.shape)

torch.Size([887518, 3])
torch.Size([887518])


In [10]:
# We also need to intialize the weights and bias, here we can use Xavier intialization for the weights, biases can be intialized to 0
# Xavier is good when we use tanh, in this case we are using it because the paper does it (Note: ReLU is more frequently used now)
# Again we are using this intialization to prevent signal blow up or vanishing
a1 = math.sqrt(6 / (embedding_space * context_length + HIDDEN_LAYER))
a2 = math.sqrt(6 / (HIDDEN_LAYER + vocab_size))
w1 = torch.rand((embedding_space * context_length, HIDDEN_LAYER)) * (2 * a1) - a1
w2 = torch.rand((HIDDEN_LAYER, vocab_size)) * (2 * a2) - a2
b1 = torch.zeros(HIDDEN_LAYER)
b2 = torch.zeros(vocab_size)

#Shape Checking
print(f"Weights 1 Shape: {w1.shape}")
print(f"Weights 2: {w2.shape}")
print(f"Bias 1: {b1.shape}")
print(f"Bias 2: {b2.shape}")

Weights 1 Shape: torch.Size([384, 700])
Weights 2: torch.Size([700, 9999])
Bias 1: torch.Size([700])
Bias 2: torch.Size([9999])


So the below actually fully runs backprop but there are many issues with it and it seems to converge with a CE Loss of around ~5.3 (Started at ~9.2) You can expirement with different learning rates and how they decay as well a maybe run more epochs.\
In the future I will implement a version using strictly PyTorch fixing some issues including:\
     - Add more data, I have around 888k words which is good but more is definetly better\
     - The lack of data causes words that don't appear frequently to have bad embeddings (<unk> is in the dataset but it's still not sufficient)\
     - I will decrease the embedding dimesion to around 30-60\
     - I will use a different optimizer (Adam) instead of this for quicker and smoother convergence times\
     - I will also increase the context window from 3 to 5
     
This is just used as a notebook to learn and fully understand all the core concepts and ideas behind word embeddings and MLP

In [12]:
## Forward pass and backprop of entire training set iterated
#Should do len(X) // 512 ----> 1733 batches if entire training set is used
batch_size = 512
batch_count = len(X_train) // batch_size
val_batch_count = len(X_val) // batch_size

#learning rate
lr = 1
lr_final = 0.005
epochs = 20

for epoch in range (epochs):
    perm = torch.randperm(X_train.shape[0])
    X_train = X_train[perm]
    Y_train = Y_train[perm]

    batch_start = 0
    batch_end = batch_size
    total_loss = 0
    for b in range(batch_count):
        #Shape: (512, 3) & (512)
        mini_batch_train = X_train[batch_start:batch_end]
        mini_batch_labels = Y_train[batch_start:batch_end]
        
        #Must convert to long or int because indices can only be integer values not floats in pytorch
        mini_batch_train = mini_batch_train.long()
        mini_batch_labels = mini_batch_labels.long()
        
        #Now we need to convert our word ---> index to their correct feature vector
        #Shape: (512, 3, 384) ----> (512, 384)
        mini_batch = C[mini_batch_train].view(mini_batch_labels.shape[0], 384)
        
        #Forward Pass now
        #Shape: (512, 384) @ (384, 700) ----> (512, 700)
        z1 = mini_batch @ w1 + b1
        a1 = torch.tanh(z1)          #tanh activation
        
        #Shape: (512, 700) @ (700, 9999) ----> (512, 9999)
        z2 = a1 @ w2 + b2
        probs = torch.softmax(z2, dim=1)    #softmaxed logits give probabilities (what our model predicts for next word)
        
        #Loss function (Cross-Entropy)
        correct_probs = probs[torch.arange(mini_batch_labels.shape[0]), mini_batch_labels]
        loss = -torch.log(correct_probs).mean()
        total_loss += loss.item()

        half_epoch = batch_count // 2
        if b != 0 and b % 200 == 0:
            print(f"Average CE Loss (Halfway through Epoch): {total_loss / half_epoch}")
        
        #Now I will do manual backprop to fully understand it all
        #First we will get ∂L/∂z2 because gradient always passes through here for all parameters
        #Shape: (512, 9999)
        dz2 = probs.clone()
        dz2[torch.arange(mini_batch_labels.shape[0]), mini_batch_labels] -= 1
        dz2 /= mini_batch_labels.shape[0]
    
        #Gradient with respect to w2 ---> ∂L/∂z2 * ∂z2/∂w2
        #∂z2/∂w2 = a1
        #Shape: (700, 512) @ (512, 9999) ----> (700, 9999), the shape must match the shape of w2
        dw2 = a1.T @ dz2 
        
        #Gradient with respect to b2 ---> ∂L/∂z2 * ∂z2/∂b2
        #∂z2/∂b2 = 1
        #Shape: (9999), we have to sum each column to match the shape of b2
        db2 = dz2.sum(axis=0)
    
        #Gradient with respect to w1 ---> ∂L/∂z2 * ∂z2/∂a1 * ∂a1/∂z1 * ∂z1/∂w1
        #∂z2/∂a1 = w2, ∂a1/∂z1 = 1 - tanh(z1)^2, ∂z1/∂w1 = mini_batch
        #(700, 9999)       (512, 700)                 (512, 384) 
        #Shape: (384, 512) @ ((512,9999) @ (9999, 700) * (512, 700)) ----> (384, 700), should match shape of w1
        da1 = dz2 @ w2.T
        dz1 = da1 * (1 - torch.tanh(z1)**2)
        dw1 = mini_batch.T @ dz1
        
        #Gradient with respect to b1 ---> ∂L/∂z2 * ∂z2/∂a1 * ∂a1/∂z1 * ∂z1/∂b1
        #∂z1/∂b1 = 1
        #Shape: (700), should match shape of b1
        db1 = dz1.sum(axis=0)
        
        #Now we need to find how the embedding matrix C affects the loss function
        #Gradient with respect to C ---> ∂L/∂z2 * ∂z2/∂a1 * ∂a1/∂z1 * ∂z1/∂mini_batch * ∂mini_batch/∂C
        #∂z1/∂mini_batch = w1
        #Shape: (512,700) @ (700, 384) ----> (512,384)
        dmini_batch = dz1 @ w1.T
        dmini_batch = dmini_batch.view(mini_batch_labels.shape[0], 3, 128)           #dmini_batch[i, j] = the gradient vector for ith training sample and the jth context word
        
        #To update C we only can change the context word parameters so we only want to change the feature vectors that were inputted as a batch into the model
        dC = torch.zeros_like(C)
        dC.index_add_(0, mini_batch_train.view(-1), dmini_batch.view(-1, 128))
        
        #Then we update our parameters, this takes step the opposite way of the steepest slope upwards
        w1 -= lr * dw1
        b1 -= lr * db1
        w2 -= lr * dw2
        b2 -= lr * db2
        C -= lr * dC

        batch_start += batch_size
        batch_end += batch_size
  
    lr -= (lr - lr_final) * (epoch / epochs)
                            
    #Now we will add the perplexity metric to model how 'confident' the model is in it's prediction
    #If the model assigns a low probability to the correct word it gets penalized very heavily
    perplexity = 0
    batch_start = 0
    batch_end = batch_size
    
    for vb in range(val_batch_count):
        mini_batch_val = X_val[batch_start:batch_end]
        mini_batch_val_labels = Y_val[batch_start:batch_end]
        
        #Must convert to long or int because indices can only be integer values not floats in pytorch
        mini_batch_val = mini_batch_val.long()
        mini_batch_val_labels = mini_batch_val_labels.long()

        #I learned this is considered the projection layer because we go from sparse vector embeddings to dense word embeddings
        #We squished dimensions from 9999 ---> 128
        mini_batch = C[mini_batch_val].view(mini_batch_val.shape[0], 384)
        
        z1 = mini_batch @ w1 + b1
        a1 = torch.tanh(z1)
        
        z2 = a1 @ w2 + b2
        probs = torch.softmax(z2, dim=1)
        
        #Loss function (Cross-Entropy)
        correct_probs = probs[torch.arange(mini_batch_val_labels.shape[0]), mini_batch_val_labels]
        loss = -torch.log(correct_probs).mean()
        perplexity += torch.exp(loss)

        batch_start += batch_size
        batch_end += batch_size
    print(f"Epoch {epoch} | Avg Train Loss: {total_loss/batch_count:.4f} | Val Perplexity: {perplexity/val_batch_count:.2f}")
        

Average CE Loss (Halfway through Epoch): 1.9210329035143832
Average CE Loss (Halfway through Epoch): 3.7724620411750385
Average CE Loss (Halfway through Epoch): 5.581840643807063
Average CE Loss (Halfway through Epoch): 7.3628738920754


KeyboardInterrupt: 